In [12]:
import pandas as pd

# Colorado sample monthly files (ingested via data/ingest_monthly_csv_to_postgres.py)
data = pd.read_csv("../data/monthly_csv/colorado_air_quality_2026_01.csv", low_memory=False)

data.head()

,Date,Hour,Station ID,Location,Offset,Pollutant,Unit,Measurement,Network,source_file,latitude,longitude
0,01/01/26,00:00,000010601,Goose Bay,-4.0,OZONE,PPB,37.0,Canadian Air and Precipitation Monitoring Network,HourlyData_2026010100.dat,40.016084,-105.270500
1,01/01/26,00:00,000020104,CHARLOTTETOWN,-4.0,NO,PPB,0.0,Canada-Prince Edward Island1,HourlyData_2026010100.dat,40.013615,-105.268844
2,01/01/26,00:00,000020104,CHARLOTTETOWN,-4.0,NO2,PPB,0.4,Canada-Prince Edward Island1,HourlyData_2026010100.dat,40.013615,-105.268844
3,01/01/26,00:00,000020104,CHARLOTTETOWN,-4.0,OZONE,PPB,38.0,Canada-Prince Edward Island1,HourlyData_2026010100.dat,40.013615,-105.268844
4,01/01/26,00:00,000020104,CHARLOTTETOWN,-4.0,PM2.5,UG/M3,3.0,Canada-Prince Edward Island1,HourlyData_2026010100.dat,40.013615,-105.268844


In [13]:
# print unique station id
unique_station_ids = data['Station ID'].unique()

print("Number of unique station ids: ", len(unique_station_ids))
print("Unique station ids: ", unique_station_ids)

Number of unique station ids:  2024
Unique station ids:  ['000010601' '000020104' '000020301' ... '840450151002' 'CC0092001'
 '840530330090']


In [14]:
# print unique "Date"
unique_dates = data['Date'].unique()

print("Number of unique dates: ", len(unique_dates))
print("Unique dates: ", unique_dates)

Number of unique dates:  31
Unique dates:  ['01/01/26' '01/02/26' '01/03/26' '01/04/26' '01/05/26' '01/06/26'
 '01/07/26' '01/08/26' '01/09/26' '01/10/26' '01/11/26' '01/12/26'
 '01/13/26' '01/14/26' '01/15/26' '01/16/26' '01/17/26' '01/18/26'
 '01/19/26' '01/20/26' '01/21/26' '01/22/26' '01/23/26' '01/24/26'
 '01/25/26' '01/26/26' '01/27/26' '01/28/26' '01/29/26' '01/30/26'
 '01/31/26']


In [16]:
# Install map widget (run once per environment)
%pip install -q ipyleaflet ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [17]:
from __future__ import annotations

import json
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display
from ipyleaflet import LayerGroup, Map, Marker, Polygon

# Paths: notebook often runs with cwd = repo/data
def _data_dir() -> Path:
    cwd = Path.cwd()
    if (cwd / "monthly_csv").exists():
        return cwd
    if (cwd / "data" / "monthly_csv").exists():
        return cwd / "data"
    return cwd


STATIONS_PATH = _data_dir() / "colorado_stations.json"

# Approximate Colorado highlight (state bbox); refine with a proper GeoJSON if you prefer
CO_RECT = Polygon(
    locations=[
        [41.0, -109.05],
        [41.0, -102.05],
        [37.0, -102.05],
        [37.0, -109.05],
    ],
    color="#1a5276",
    weight=2,
    fill_color="#2980b9",
    fill_opacity=0.12,
)

stations: list[dict] = []
station_seq = {"n": 0}
markers_layer = LayerGroup()

m = Map(center=(39.0, -105.5), zoom=6, scroll_wheel_zoom=True)
m.add_layer(CO_RECT)
m.add_layer(markers_layer)


def _next_station_id() -> str:
    station_seq["n"] += 1
    # 9-digit numeric ids (800xxxxxx block for this synthetic workflow)
    return f"{800_000_000 + station_seq['n']:09d}"


def on_map_click(**kwargs):
    if kwargs.get("type") != "click":
        return
    coords = kwargs.get("coordinates")
    if not coords:
        return
    lat, lon = float(coords[0]), float(coords[1])
    sid = _next_station_id()
    name = f"CO_STATION_{station_seq['n']:03d}"
    stations.append({"station_id": sid, "location": name, "latitude": lat, "longitude": lon})
    markers_layer.add_layer(Marker(location=(lat, lon), title=f"{sid} {name}"))
    status.value = f"{len(stations)} station(s). Last: {sid} @ ({lat:.5f}, {lon:.5f})"


m.on_interaction(on_map_click)

status = widgets.HTML(value="Click inside the highlighted region to add stations.")

btn_clear = widgets.Button(description="Clear stations", button_style="warning")
btn_save = widgets.Button(description="Save stations to JSON", button_style="success")
btn_load = widgets.Button(description="Load stations from JSON")


def on_clear(_):
    stations.clear()
    station_seq["n"] = 0
    markers_layer.clear_layers()
    status.value = "Cleared. Click map to add stations."


def on_save(_):
    STATIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
    STATIONS_PATH.write_text(json.dumps(stations, indent=2), encoding="utf-8")
    status.value = f"Saved {len(stations)} station(s) to {STATIONS_PATH}"


def on_load(_):
    if not STATIONS_PATH.exists():
        status.value = f"No file at {STATIONS_PATH}"
        return
    loaded = json.loads(STATIONS_PATH.read_text(encoding="utf-8"))
    stations.clear()
    markers_layer.clear_layers()
    max_id = 800_000_000
    for row in loaded:
        stations.append(row)
        max_id = max(max_id, int(row["station_id"]))
        lat, lon = row["latitude"], row["longitude"]
        markers_layer.add_layer(
            Marker(location=(lat, lon), title=f"{row['station_id']} {row['location']}")
        )
    # Next click uses max_id + 1 (counter = suffix after 800000000)
    station_seq["n"] = max(0, max_id - 800_000_000)
    status.value = f"Loaded {len(stations)} station(s) from {STATIONS_PATH}"


btn_clear.on_click(on_clear)
btn_save.on_click(on_save)
btn_load.on_click(on_load)

display(widgets.VBox([widgets.HBox([btn_clear, btn_save, btn_load]), status, m]))

In [2]:
import json
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd


def _data_dir() -> Path:
    cwd = Path.cwd()
    if (cwd / "monthly_csv").exists():
        return cwd
    if (cwd / "data" / "monthly_csv").exists():
        return cwd / "data"
    return cwd


STATIONS_PATH = _data_dir() / "colorado_stations.json"
OUT_DIR = _data_dir() / "monthly_csv"
NETWORK = "Colorado Synthetic Sample"
OFFSET = -7.0

# (name, unit, baseline, phi_dev, sigma, vmin, vmax, episode_strength)
# baseline + small AR noise = mostly WHO "good". episode_strength scales 1–2 built-in events
# (ramp → flat plateau → decay) so charts show realistic peaks without everything being a sine wave.
POLLUTANT_SPEC = [
    ("PM2.5", "UG/M3", 8.5, 0.78, 1.05, 0.0, 95.0, 42.0),
    ("PM10", "UG/M3", 24.0, 0.76, 3.2, 0.0, 220.0, 88.0),
    ("OZONE", "PPB", 30.0, 0.80, 2.2, 4.0, 98.0, 28.0),
    ("SO2", "PPB", 1.2, 0.72, 0.45, 0.0, 45.0, 9.0),
    ("NO", "PPB", 10.5, 0.78, 1.45, 0.0, 75.0, 22.0),
    ("NO2", "PPB", 11.5, 0.79, 1.65, 0.0, 92.0, 26.0),
]


def _episode_envelope(n_hours: int, rng: np.random.Generator) -> np.ndarray:
    """1–2 non-overlapping trapezoid episodes: rise, plateau, fall (0..1)."""
    env = np.zeros(n_hours, dtype=np.float64)
    n_ep = int(rng.integers(1, 3))
    used = np.zeros(n_hours, dtype=bool)
    for _ in range(n_ep):
        for _try in range(80):
            total_w = int(rng.integers(40, 100))
            rise = int(rng.integers(6, 20))
            fall = int(rng.integers(6, 20))
            plateau = max(16, total_w - rise - fall)
            total_w = rise + plateau + fall
            latest_start = n_hours - total_w
            if latest_start < 1:
                continue
            start = int(rng.integers(0, latest_start))
            if np.any(used[start : start + total_w]):
                continue
            for k in range(total_w):
                idx = start + k
                if k < rise:
                    w = (k + 1) / rise
                elif k < rise + plateau:
                    w = 1.0
                else:
                    k2 = k - rise - plateau
                    w = max(0.0, 1.0 - (k2 + 1) / max(1, fall))
                env[idx] = np.maximum(env[idx], w)
            used[start : start + total_w] = True
            break
    return env


def _synthetic_hourly_series(
    n_hours: int,
    baseline: float,
    phi_dev: float,
    sigma: float,
    vmin: float,
    vmax: float,
    episode_strength: float,
    rng: np.random.Generator,
) -> np.ndarray:
    env = _episode_envelope(n_hours, rng)
    x = np.empty(n_hours, dtype=np.float64)
    dev = 0.0
    for idx in range(n_hours):
        dev = phi_dev * dev + float(rng.normal(0.0, sigma * 0.9))
        micro = float(rng.normal(0.0, sigma * 0.38))
        plate_jitter = 1.0 + 0.08 * float(rng.standard_normal()) if env[idx] > 0.85 else 1.0
        extra = episode_strength * env[idx] * plate_jitter
        raw = baseline + dev + micro + extra
        x[idx] = float(np.clip(raw, vmin, vmax))
    return x


def _source_file(dt: datetime) -> str:
    return f"HourlyData_{dt:%Y%m%d%H}.dat"


if not STATIONS_PATH.exists():
    raise FileNotFoundError(
        f"Missing {STATIONS_PATH}. Run the map cell, click stations, then Save stations to JSON."
    )

stations = json.loads(STATIONS_PATH.read_text(encoding="utf-8"))
if not stations:
    raise ValueError("colorado_stations.json is empty — add at least one station on the map and save.")

t0 = datetime(2026, 1, 1, 0, 0)
t1 = datetime(2026, 5, 1, 0, 0)  # exclusive end → includes all of Apr 2026
n_hours = int((t1 - t0).total_seconds() // 3600)

# Precompute series per station × pollutant (slight per-station baseline tilt)
series_cache: dict[tuple[str, str], np.ndarray] = {}
for st in stations:
    sid = str(st["station_id"]).zfill(9)
    st_id = int(sid)
    station_bias = 0.96 + 0.10 * ((st_id % 11) / 10.0)
    for j, row in enumerate(POLLUTANT_SPEC):
        pname, _unit, baseline, phi_dev, sigma, vmin, vmax, ep_strength = row
        seed = (st_id * 1_000_003 + j * 97 + sum(ord(c) for c in pname)) & 0xFFFFFFFF
        rng = np.random.default_rng(seed)
        bl = baseline * station_bias
        series_cache[(sid, pname)] = _synthetic_hourly_series(
            n_hours,
            bl,
            phi_dev,
            sigma,
            vmin,
            vmax,
            ep_strength,
            rng,
        )

rows_by_month: dict[int, list[dict]] = {1: [], 2: [], 3: [], 4: []}

for h in range(n_hours):
    dt = t0 + timedelta(hours=h)
    month = dt.month
    if month not in rows_by_month:
        continue
    date_str = dt.strftime("%m/%d/%y")
    hour_str = dt.strftime("%H:%M")
    src = _source_file(dt)
    for st in stations:
        sid = str(st["station_id"]).zfill(9)
        loc = st["location"]
        lat = st["latitude"]
        lon = st["longitude"]
        for pname, unit, *_rest in POLLUTANT_SPEC:
            val = float(series_cache[(sid, pname)][h])
            measurement = round(val, 1)
            rows_by_month[month].append(
                {
                    "Date": date_str,
                    "Hour": hour_str,
                    "Station ID": sid,
                    "Location": loc,
                    "Offset": OFFSET,
                    "Pollutant": pname,
                    "Unit": unit,
                    "Measurement": measurement,
                    "Network": NETWORK,
                    "source_file": src,
                    "latitude": lat,
                    "longitude": lon,
                }
            )

OUT_DIR.mkdir(parents=True, exist_ok=True)
for mnum, fname in [(1, "colorado_air_quality_2026_01.csv"), (2, "colorado_air_quality_2026_02.csv"), (3, "colorado_air_quality_2026_03.csv"), (4, "colorado_air_quality_2026_04.csv")]:
    df = pd.DataFrame(rows_by_month[mnum])
    path = OUT_DIR / fname
    df.to_csv(path, index=False)
    print(path, "rows:", len(df))

df.head(12)


c:\Users\Aditya\Documents\Masters\DataCenter\AirTrail\data\monthly_csv\colorado_air_quality_2026_01.csv rows: 44640
c:\Users\Aditya\Documents\Masters\DataCenter\AirTrail\data\monthly_csv\colorado_air_quality_2026_02.csv rows: 40320
c:\Users\Aditya\Documents\Masters\DataCenter\AirTrail\data\monthly_csv\colorado_air_quality_2026_03.csv rows: 44640
c:\Users\Aditya\Documents\Masters\DataCenter\AirTrail\data\monthly_csv\colorado_air_quality_2026_04.csv rows: 43200


,Date,Hour,Station ID,Location,Offset,Pollutant,Unit,Measurement,Network,source_file,latitude,longitude
0,04/01/26,00:00,800000001,CO_STATION_001,-7.0,PM2.5,UG/M3,10.1,Colorado Synthetic Sample,HourlyData_2026040100.dat,40.051166,-105.270922
1,04/01/26,00:00,800000001,CO_STATION_001,-7.0,PM10,UG/M3,21.4,Colorado Synthetic Sample,HourlyData_2026040100.dat,40.051166,-105.270922
2,04/01/26,00:00,800000001,CO_STATION_001,-7.0,OZONE,PPB,31.0,Colorado Synthetic Sample,HourlyData_2026040100.dat,40.051166,-105.270922
3,04/01/26,00:00,800000001,CO_STATION_001,-7.0,SO2,PPB,1.2,Colorado Synthetic Sample,HourlyData_2026040100.dat,40.051166,-105.270922
4,04/01/26,00:00,800000001,CO_STATION_001,-7.0,NO,PPB,10.5,Colorado Synthetic Sample,HourlyData_2026040100.dat,40.051166,-105.270922
5,04/01/26,00:00,800000001,CO_STATION_001,-7.0,NO2,PPB,14.5,Colorado Synthetic Sample,HourlyData_2026040100.dat,40.051166,-105.270922
6,04/01/26,00:00,800000002,CO_STATION_002,-7.0,PM2.5,UG/M3,8.4,Colorado Synthetic Sample,HourlyData_2026040100.dat,40.059049,-105.324465
7,04/01/26,00:00,800000002,CO_STATION_002,-7.0,PM10,UG/M3,62.2,Colorado Synthetic Sample,HourlyData_2026040100.dat,40.059049,-105.324465
8,04/01/26,00:00,800000002,CO_STATION_002,-7.0,OZONE,PPB,35.1,Colorado Synthetic Sample,HourlyData_2026040100.dat,40.059049,-105.324465
9,04/01/26,00:00,800000002,CO_STATION_002,-7.0,SO2,PPB,2.1,Colorado Synthetic Sample,HourlyData_2026040100.dat,40.059049,-105.324465
